# Transformation pipelines and custom transformers

**Author:** [firstname] [lastname]

COSC 3430 Artificial Intelligence | Fall 26 | Augustana U.

End-to-End ML Project, part 7. Code along with the slides: each
section below matches one code-along in the slide deck.

Code-alongs 1-5 and 8-11 are filled in. Code-alongs 6, 7, 12 and 13
are empty: we code them together in class.

Run the setup cell first. It loads the housing data and rebuilds
`housing`, `housing_labels` and `housing_num` from part 6. Then run
the cells in order. Code-along 8 uses `num_pipeline` and `cat_pipeline` from
code-alongs 6 and 7, so code them before you run 8.

In [ ]:
# Setup: load the housing data and rebuild the training set from part 6
# data frames and arrays
import pandas as pd
import numpy as np
# train/test split
from sklearn.model_selection import train_test_split
# one-hot encoding of categories
from sklearn.preprocessing import OneHotEncoder

# load the California housing data
housing_full = pd.read_csv("https://tinyurl.com/housing-csv")
# income categories for stratified sampling
housing_full["income_cat"] = pd.cut(
    housing_full["median_income"],
    bins=[0, 1.5, 3.0, 4.5, 6.0, np.inf],
    labels=[1, 2, 3, 4, 5])
# 80/20 split that keeps the income mix in both sets
strat_train_set, strat_test_set = train_test_split(
    housing_full, test_size=0.2,
    stratify=housing_full["income_cat"],
    random_state=42)
# drop the helper column from both sets
for df in (strat_train_set, strat_test_set):
    df.drop("income_cat", axis=1, inplace=True)
# predictors: the training set without the target
housing = strat_train_set.drop("median_house_value", axis=1)
# labels: a copy of the target column
housing_labels = strat_train_set["median_house_value"].copy()
# numerical features only
housing_num = housing.select_dtypes(include=[np.number])
# check the dimensions
print(housing.shape, housing_num.shape)

Expected output:

```
(16512, 9) (16512, 8)
```

## Code-along 1. A small example

In [ ]:
# Code-along 1: a toy data set with missing values
# two numeric columns, one NaN each
toy = pd.DataFrame({
    "age":    [25, 32, np.nan, 40, 28],
    "income": [50000, 64000, 58000, np.nan, 72000],
})
# the data
print(toy)
# summary statistics
print(toy.describe())

## Code-along 2. `Pipeline`: impute, then scale

In [ ]:
# Code-along 2: a pipeline that imputes, then scales
# import the pipeline and the two transformers
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# two named steps, applied in this order
num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

# fit and transform all steps in one call
toy_prepared = num_pipeline.fit_transform(toy)
# print rounded to two decimals
print(toy_prepared.round(2))

## Code-along 3. The same steps by hand

In [ ]:
# Code-along 3: the same two steps by hand
# step 1: impute the medians
imputer = SimpleImputer(strategy="median")
toy_imputed = imputer.fit_transform(toy)
# no NaN left
print(toy_imputed)

# step 2: standardize the imputed data
scaler = StandardScaler()
toy_scaled = scaler.fit_transform(toy_imputed)
# same result as the pipeline
print(np.round(toy_scaled, 2))

## Code-along 4. A predictor as the last step

In [ ]:
# Code-along 4: a pipeline that ends with a predictor
# import the model and the error metric
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

# keep only the rows with a known target
train = toy.dropna(subset=["income"])
# feature as a 2D DataFrame
X = train[["age"]]
# target as a Series
y = train["income"]

# impute, scale, then predict
full_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("predict", LinearRegression())
])

# fit all three steps
full_pipeline.fit(X, y)
# predict on the same rows
predictions = full_pipeline.predict(X)
# rounded predictions
print(np.round(predictions, 0))
# root mean squared error
rmse = root_mean_squared_error(y, predictions)
# print with a thousands separator
print(f"RMSE: {rmse:,.0f}")

## Code-along 5. `make_pipeline`, and looking inside

In [ ]:
# Code-along 5: a pipeline without step names
# import the helper
from sklearn.pipeline import make_pipeline

# the same three steps, named automatically
pipeline = make_pipeline(SimpleImputer(strategy="median"),
                         StandardScaler(),
                         LinearRegression())

In [ ]:
# Code-along 5 (cont.): look inside the pipeline
# whole pipeline
print(pipeline)
# just the steps
print(pipeline.steps)
# just the length
print(len(pipeline))
# element using index
print(pipeline[0])
# last element
print(pipeline[-1])

## Code-along 6. A pipeline for the housing data

## Code-along 7. `ColumnTransformer`: numerical and categorical columns

## Code-along 8. Let scikit-learn pick the columns

In [ ]:
# Code-along 8: select the columns by data type
# import the two helpers
from sklearn.compose import make_column_selector, make_column_transformer

# numerical pipeline on number columns, categorical on text columns
preprocessing = make_column_transformer(
    (num_pipeline, make_column_selector(dtype_include=np.number)),
    (cat_pipeline, make_column_selector(dtype_include=object)),
)
# fit and transform the whole training set
housing_prepared = preprocessing.fit_transform(housing)
# 8 numerical + 5 one-hot columns
print(housing_prepared.shape)
# the generated feature names
print(preprocessing.get_feature_names_out())

## Code-along 9. A custom transformer: `ClusterSimilarity`

In [ ]:
# Code-along 9: a transformer class of our own
# base classes for estimators and transformers
from sklearn.base import BaseEstimator, TransformerMixin
# Gaussian RBF
from sklearn.metrics.pairwise import rbf_kernel
# k-means clustering
from sklearn.cluster import KMeans

# inherits get_params(), set_params() and fit_transform()
class ClusterSimilarity(BaseEstimator, TransformerMixin):
    # store the hyperparameters, nothing else
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        # number of clusters
        self.n_clusters = n_clusters
        # RBF decay
        self.gamma = gamma
        # seed for KMeans
        self.random_state = random_state

    # learn the cluster centers from the training data
    def fit(self, X, y=None, sample_weight=None):
        # a KMeans model with n_clusters clusters
        self.kmeans_ = KMeans(self.n_clusters, random_state=self.random_state)
        # find the clusters
        self.kmeans_.fit(X, sample_weight=sample_weight)
        # always return self!
        return self

    # similarity of each row to each cluster center
    def transform(self, X):
        # one column per cluster
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    # names of the output columns
    def get_feature_names_out(self, names=None):
        # one name per cluster
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

## Code-along 10. Use it like any other transformer

In [ ]:
# Code-along 10: cluster similarities for latitude and longitude
# 10 clusters, gamma = 1
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
# fit the clusters, then compute the similarities
similarities = cluster_simil.fit_transform(housing[["latitude", "longitude"]])
# one column per cluster
print(similarities.shape)
# first three districts
print(similarities[:3].round(2))

## Code-along 11. Reusable pieces: `FunctionTransformer`

In [ ]:
# Code-along 11: the building blocks of the final transformer
# wraps a function as a transformer
from sklearn.preprocessing import FunctionTransformer

# ratio of the first to the second column
def column_ratio(X):
    # keep 2D shape with [[0]] and [[1]]
    return X[:, [0]] / X[:, [1]]

# feature name of the ratio column
def ratio_name(function_transformer, feature_names_in):
    # one output column called "ratio"
    return ["ratio"]

# impute, take the ratio, scale
def ratio_pipeline():
    # a new pipeline at every call
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler())

# impute, take the log, scale
log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler())

# 10 cluster similarities
cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)

# impute and scale everything else
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"),
                                     StandardScaler())

## Code-along 12. The final `ColumnTransformer`

## Code-along 13. 24 features, ready for training